# Банковский консультант: RAG-система для автоматизации ответов клиентам

Ноутбук предназначен для Google Colab. Он создает синтетическую базу банковских документов, строит векторный индекс Chroma, сравнивает retrieval-стратегии и запускает RAG-цепочку с OpenAI при наличии `OPENAI_API_KEY`.

Ключ API не хранится в ноутбуке: введите его через защищенный prompt или заранее задайте переменную окружения.

In [ ]:
%pip install -q -U langchain langchain-openai langchain-community langchain-chroma chromadb sentence-transformers rank-bm25 ragas datasets openai tiktoken unstructured[pdf]

In [ ]:
import os
import time
from functools import lru_cache
from getpass import getpass
from typing import Callable, Dict, List, Optional, Tuple, Union

import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory
from langchain.retrievers import ContextualCompressionRetriever, MultiQueryRetriever
from langchain.retrievers.document_compressors import LLMChainExtractor
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter,
    SentenceTransformersTokenTextSplitter,
)

try:
    from datasets import Dataset
    from ragas import evaluate
    from ragas.metrics import answer_relevancy, context_precision, faithfulness
except Exception as exc:  # Ragas API can differ between versions; handle it where used.
    Dataset = None
    evaluate = None
    answer_relevancy = None
    context_precision = None
    faithfulness = None
    print(f"Ragas недоступен: {exc}")

if not os.getenv("OPENAI_API_KEY"):
    api_key = getpass("OPENAI_API_KEY (оставьте пустым, чтобы пропустить LLM/Ragas шаги): ").strip()
    if api_key:
        os.environ["OPENAI_API_KEY"] = api_key

HAS_OPENAI_KEY = bool(os.getenv("OPENAI_API_KEY"))
print("OpenAI key configured:", HAS_OPENAI_KEY)

## 1. Синтетические документы

In [ ]:
def create_synthetic_docs() -> List[Dict[str, str]]:
    return [
        {
            "id": "doc1",
            "product_type": "credit",
            "section": "terms",
            "text": """
            Кредит наличными. Процентная ставка от 12% до 25% годовых в зависимости от кредитной истории.
            Сумма кредита от 50 000 до 5 000 000 рублей. Срок кредита от 6 до 60 месяцев.
            Требования к заёмщику: возраст от 21 до 65 лет, стаж на последнем месте работы не менее 6 месяцев,
            подтверждение дохода справкой 2-НДФЛ или выпиской по счёту.
            Досрочное погашение без комиссии через 6 месяцев после выдачи.
            """,
        },
        {
            "id": "doc2",
            "product_type": "mortgage",
            "section": "terms",
            "text": """
            Ипотечное кредитование. Первоначальный взнос от 15% стоимости недвижимости.
            Процентная ставка от 8% до 15% годовых в зависимости от программы и размера взноса.
            Срок кредита до 30 лет. Обязательное страхование жизни и здоровья заёмщика и объекта недвижимости.
            Требования к заёмщику: возраст от 21 до 65 лет, стаж от 1 года на последнем месте.
            """,
        },
        {
            "id": "doc3",
            "product_type": "deposit",
            "section": "rates",
            "text": """
            Депозиты. Виды вкладов: «Классический» (ставка 6% годовых), «Накопительный» (ставка 7% годовых),
            «Инвестиционный» (ставка до 9% годовых). Минимальная сумма вклада: 10 000 рублей.
            Срок вклада от 3 до 36 месяцев. Досрочное снятие возможно, но проценты пересчитываются по ставке до востребования (1%).
            Пополнение и частичное снятие разрешены только для «Накопительного» вклада.
            """,
        },
        {
            "id": "doc4",
            "product_type": "card",
            "section": "terms",
            "text": """
            Кредитные карты. Льготный период до 55 дней. Процентная ставка от 18% до 30% годовых.
            Кредитный лимит до 500 000 рублей. Комиссия за снятие наличных 3% (мин. 300 руб.).
            Бесплатное обслуживание при ежемесячных покупках от 30 000 руб., иначе 500 руб./мес.
            Требования к заёмщику: возраст от 21 до 65 лет, постоянная регистрация.
            """,
        },
        {
            "id": "doc5",
            "product_type": "faq",
            "section": "general",
            "text": """
            FAQ (часто задаваемые вопросы).
            В: Можно ли досрочно погасить кредит без комиссии?
            О: Да, через 6 месяцев после выдачи кредита, если это указано в договоре.
            В: Какие документы нужны для ипотеки?
            О: Паспорт, СНИЛС, справка о доходах (2-НДФЛ или выписка), документы на недвижимость.
            В: Какой минимальный взнос по ипотеке?
            О: 15% от стоимости квартиры.
            В: Можно ли снять деньги с депозита досрочно?
            О: Да, но проценты будут пересчитаны по ставке 1% годовых.
            """,
        },
    ]


docs = create_synthetic_docs()
print("Создано документов:", len(docs))

## 2. Чанкинг

In [ ]:
def chunk_documents(docs: List[Dict[str, str]], strategy: str = "recursive") -> List[Document]:
    if strategy == "fixed":
        splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    elif strategy == "sentence":
        splitter = SentenceTransformersTokenTextSplitter(chunk_size=500, chunk_overlap=50)
    else:
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=500,
            chunk_overlap=50,
            separators=["\n\n", "\n", ". ", " ", ""],
        )

    documents: List[Document] = []
    for doc in docs:
        chunks = splitter.split_text(doc["text"].strip())
        for chunk_idx, chunk in enumerate(chunks):
            documents.append(
                Document(
                    page_content=chunk,
                    metadata={
                        "chunk_id": f"{doc['id']}-{chunk_idx}",
                        "doc_id": doc["id"],
                        "product_type": doc["product_type"],
                        "section": doc["section"],
                    },
                )
            )
    return documents


chunks = chunk_documents(docs, strategy="recursive")
print(f"Создано чанков: {len(chunks)}")

## 3. Эмбеддинги и Chroma

In [ ]:
EMBEDDING_MODEL_NAME = os.getenv("EMBEDDING_MODEL_NAME", "intfloat/multilingual-e5-large")
PERSIST_DIRECTORY = os.getenv("CHROMA_PERSIST_DIRECTORY", "./chroma_db")

embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL_NAME)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=PERSIST_DIRECTORY,
)
print(f"Векторная БД создана в {PERSIST_DIRECTORY!r} на модели {EMBEDDING_MODEL_NAME!r}.")

## 4. Retrieval: similarity, MMR, гибридный BM25 + vector search

In [ ]:
retriever_sim = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 5})
retriever_mmr = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": 5, "fetch_k": 20})

corpus_texts = [chunk.page_content for chunk in chunks]
tokenized_corpus = [text.lower().split() for text in corpus_texts]
bm25 = BM25Okapi(tokenized_corpus)


def _doc_key(doc: Document) -> str:
    return str(doc.metadata.get("chunk_id") or f"{doc.metadata.get('doc_id')}::{doc.page_content}")


def _reciprocal_rank_fusion(rankings: List[List[Document]], k: int, rank_constant: int = 60) -> List[Document]:
    docs_by_key: Dict[str, Document] = {}
    scores: Dict[str, float] = {}

    for ranking in rankings:
        for rank, doc in enumerate(ranking):
            key = _doc_key(doc)
            docs_by_key[key] = doc
            scores[key] = scores.get(key, 0.0) + 1.0 / (rank + rank_constant)

    return [docs_by_key[key] for key in sorted(scores, key=scores.get, reverse=True)[:k]]


def hybrid_search(query: str, k: int = 5, product_type: Optional[str] = None) -> List[Document]:
    filter_dict = {"product_type": product_type} if product_type else None
    vector_docs = vectorstore.similarity_search(query, k=k * 2, filter=filter_dict)

    tokenized_query = query.lower().split()
    bm25_scores = bm25.get_scores(tokenized_query)
    candidate_indices = list(range(len(chunks)))
    if product_type:
        candidate_indices = [
            idx for idx in candidate_indices if chunks[idx].metadata.get("product_type") == product_type
        ]
    top_bm25_indices = sorted(candidate_indices, key=lambda idx: bm25_scores[idx], reverse=True)[: k * 2]
    bm25_docs = [chunks[idx] for idx in top_bm25_indices]

    return _reciprocal_rank_fusion([vector_docs, bm25_docs], k=k)


print("Пример гибридного поиска:")
for doc in hybrid_search("Какая ставка по ипотеке?", k=3):
    print(doc.metadata, doc.page_content[:120].replace("\n", " "))

## 5. Оценка качества retrieval

In [ ]:
test_queries = [
    ("Какая процентная ставка по кредиту наличными?", "doc1"),
    ("Какой минимальный первоначальный взнос по ипотеке?", "doc2"),
    ("Можно ли пополнять депозит?", "doc3"),
    ("Какой льготный период по кредитной карте?", "doc4"),
    ("Какие документы нужны для ипотеки?", "doc5"),
]


RetrieverLike = Union[Callable[[str, int], List[Document]], object]


def retrieve(retriever_func: RetrieverLike, question: str, k: int = 5) -> List[Document]:
    if callable(retriever_func):
        return retriever_func(question, k=k)
    if hasattr(retriever_func, "invoke"):
        return retriever_func.invoke(question)
    return retriever_func.get_relevant_documents(question)


def evaluate_retriever(retriever_func: RetrieverLike, queries: List[Tuple[str, str]], k: int = 5) -> Tuple[float, float]:
    hits = 0
    reciprocal_ranks = []
    for question, expected_doc_id in queries:
        retrieved = retrieve(retriever_func, question, k=k)
        for idx, doc in enumerate(retrieved):
            if doc.metadata.get("doc_id") == expected_doc_id:
                hits += 1
                reciprocal_ranks.append(1 / (idx + 1))
                break
        else:
            reciprocal_ranks.append(0)
    return hits / len(queries), float(np.mean(reciprocal_ranks))


for name, retriever in [("similarity", retriever_sim), ("mmr", retriever_mmr), ("hybrid", hybrid_search)]:
    hit, mrr = evaluate_retriever(retriever, test_queries)
    print(f"{name}: Hit@5={hit:.2f}, MRR={mrr:.2f}")

## 6. LLM и RAG-цепочка

In [ ]:
system_prompt = """
Вы - профессиональный консультант банка. Отвечайте на вопросы клиентов, используя только информацию из приведенного контекста.
Если контекст не содержит ответа, скажите: "Информации по вашему вопросу в текущей базе знаний нет. Пожалуйста, обратитесь в службу поддержки."
Всегда указывайте источник (id документа и раздел) после каждого факта. Ответ должен быть кратким, но полным.

Контекст:
{context}

Вопрос: {question}
Ответ:
""".strip()

qa_prompt = PromptTemplate.from_template(system_prompt)

llm = None
compression_retriever = retriever_sim
qa_chain = None

if HAS_OPENAI_KEY:
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2, max_tokens=1000)
    compressor = LLMChainExtractor.from_llm(llm)
    compression_retriever = ContextualCompressionRetriever(
        base_compressor=compressor,
        base_retriever=retriever_sim,
    )
    memory = ConversationBufferMemory(
        memory_key="chat_history",
        return_messages=True,
        output_key="answer",
    )
    qa_chain = ConversationalRetrievalChain.from_llm(
        llm=llm,
        retriever=compression_retriever,
        memory=memory,
        return_source_documents=True,
        combine_docs_chain_kwargs={"prompt": qa_prompt},
        verbose=False,
    )
else:
    print("LLM-цепочка пропущена: задайте OPENAI_API_KEY и перезапустите эту ячейку.")

In [ ]:
def answer_question(question: str) -> Tuple[str, List[Document]]:
    if qa_chain is None:
        raise RuntimeError("OPENAI_API_KEY не задан, RAG-цепочка с LLM не инициализирована.")

    result = qa_chain.invoke({"question": question})
    answer_text = result["answer"]
    sources = result.get("source_documents", [])

    if sources:
        source_refs = sorted(
            {
                f"{doc.metadata.get('doc_id', 'unknown')} ({doc.metadata.get('section', 'unknown')})"
                for doc in sources
            }
        )
        answer_text += "\n\nИсточники: " + ", ".join(source_refs)
    return answer_text, sources


if qa_chain is not None:
    test_q = "Какая ставка по ипотеке?"
    ans, src = answer_question(test_q)
    print("Вопрос:", test_q)
    print("Ответ:", ans)

## 7. Оптимизация: multi-query и reranking

In [ ]:
multi_retriever = MultiQueryRetriever.from_llm(retriever=retriever_sim, llm=llm) if llm else None
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")


def rerank_docs(query: str, docs: List[Document], top_k: int = 5) -> List[Document]:
    if not docs:
        return []
    pairs = [[query, doc.page_content] for doc in docs]
    scores = cross_encoder.predict(pairs)
    sorted_idx = np.argsort(scores)[::-1][:top_k]
    return [docs[idx] for idx in sorted_idx]


class RerankRetriever:
    def __init__(self, base_retriever, top_k: int = 5):
        self.base_retriever = base_retriever
        self.top_k = top_k

    def invoke(self, query: str, config=None) -> List[Document]:
        docs = retrieve(self.base_retriever, query, k=self.top_k * 2)
        return rerank_docs(query, docs, top_k=self.top_k)

    def get_relevant_documents(self, query: str) -> List[Document]:
        return self.invoke(query)


rerank_retriever = RerankRetriever(retriever_sim, top_k=5)
print("Rerank retriever ready:", len(rerank_retriever.invoke("ставка по кредиту")))

## 8. Ragas-оценка качества ответов

In [ ]:
def run_ragas_evaluation():
    if qa_chain is None:
        print("Ragas пропущен: RAG-цепочка не инициализирована.")
        return None
    if not all([Dataset, evaluate, faithfulness, answer_relevancy, context_precision]):
        print("Ragas пропущен: библиотека или метрики недоступны.")
        return None

    test_questions = [
        "Какая процентная ставка по кредиту наличными?",
        "Какой минимальный взнос по ипотеке?",
        "Можно ли пополнять депозит?",
    ]
    ground_truths = [
        "От 12% до 25% годовых.",
        "15% от стоимости недвижимости.",
        "Да, для накопительного вклада.",
    ]

    answers = []
    contexts = []
    for question in test_questions:
        answer, source_docs = answer_question(question)
        answers.append(answer)
        contexts.append([doc.page_content for doc in source_docs])

    dataset = Dataset.from_dict(
        {
            "question": test_questions,
            "answer": answers,
            "contexts": contexts,
            "ground_truth": ground_truths,
        }
    )
    return evaluate(dataset, metrics=[faithfulness, answer_relevancy, context_precision])


ragas_result = run_ragas_evaluation()
if ragas_result is not None:
    print(ragas_result)

## 9. Производительность: время ответа, LRU-кеш и батчи

In [ ]:
def measure_time(func, *args, **kwargs):
    start = time.time()
    result = func(*args, **kwargs)
    elapsed = time.time() - start
    return result, elapsed


@lru_cache(maxsize=128)
def cached_answer(question: str) -> str:
    answer, _ = answer_question(question)
    return answer


async def batch_answers(questions: List[str]) -> List[str]:
    # Для OpenAI batch/async можно заменить на ainvoke; этот вариант сохраняет простоту Colab-демо.
    return [answer_question(question)[0] for question in questions]


if qa_chain is not None:
    q = "Что такое досрочное погашение?"
    _, time_no_cache = measure_time(answer_question, q)
    _, time_cache = measure_time(cached_answer, q)
    print(f"Время без кеша: {time_no_cache:.3f} сек")
    print(f"Время с кешем: {time_cache:.3f} сек")
else:
    print("Замеры LLM-ответов пропущены: задайте OPENAI_API_KEY.")

print("RAG-система готова. Пример: answer_question('Какой срок кредита?')")